# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [3]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [6]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif

num_col = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'] #define the columns
for col in num_col:
    spaceship[col] = spaceship[col].fillna(spaceship[col].median()) #clean the nuls

# split in Y and X
X = spaceship[num_col]
y = spaceship['Transported'].astype(int)

#feature scaling
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print("Feature scaling")
print(X_scaled.head())

#feature selection
k_best_selector = SelectKBest(score_func=f_classif, k=4)
X_selected = k_best_selector.fit_transform(X_scaled, y)
selected_features = X_scaled.columns[k_best_selector.get_support()]

print("Feature selection")
print(list(selected_features))

df_final = pd.DataFrame(X_selected, columns=selected_features)

Feature scaling
        Age  RoomService  FoodCourt  ShoppingMall       Spa    VRDeck
0  0.493671     0.000000   0.000000      0.000000  0.000000  0.000000
1  0.303797     0.007608   0.000302      0.001064  0.024500  0.001823
2  0.734177     0.003001   0.119948      0.000000  0.299670  0.002030
3  0.417722     0.000000   0.043035      0.015793  0.148563  0.007997
4  0.202532     0.021149   0.002348      0.006428  0.025214  0.000083
Feature selection
['Age', 'RoomService', 'Spa', 'VRDeck']


**Perform Train Test Split**

In [8]:
from sklearn.model_selection import train_test_split

#80% for train and 20% for test
X_train, X_test, y_train, y_test = train_test_split(
    df_final, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"X_train : {X_train.shape}")
print(f"X_test :        {X_test.shape}")
print(f"y_train:                {y_train.shape}")
print(f"y_test:                 {y_test.shape}")

X_train : (6954, 4)
X_test :        (1739, 4)
y_train:                (6954,)
y_test:                 (1739,)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [10]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

#baging model
base_estimator = DecisionTreeClassifier(random_state=42)
bagging_model = BaggingClassifier(
    estimator=base_estimator,
    n_estimators=100,      
    max_samples=0.8,       
    bootstrap=True,        
    random_state=42,
    n_jobs=-1              
)

bagging_model.fit(X_train, y_train)
y_pred_bagging = bagging_model.predict(X_test)
acc_bagging = accuracy_score(y_test, y_pred_bagging)


#pasting model
pasting_model = BaggingClassifier(
    estimator=base_estimator,
    n_estimators=100,
    max_samples=0.8,
    bootstrap=False,       
    random_state=42,
    n_jobs=-1
)

pasting_model.fit(X_train, y_train)
y_pred_pasting = pasting_model.predict(X_test)
acc_pasting = accuracy_score(y_test, y_pred_pasting)

#displau results
print("Accuracy of the models")
print(f"Accuracy of Bagging: {acc_bagging:.4f}")
print(f"Accuracy of Pasting: {acc_pasting:.4f}\n")

print("Report Bagging")
print(classification_report(y_test, y_pred_bagging))

Accuracy of the models
Accuracy of Bagging: 0.7568
Accuracy of Pasting: 0.7361

Report Bagging
              precision    recall  f1-score   support

           0       0.78      0.71      0.74       863
           1       0.74      0.81      0.77       876

    accuracy                           0.76      1739
   macro avg       0.76      0.76      0.76      1739
weighted avg       0.76      0.76      0.76      1739



- Random Forests

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf_model = RandomForestClassifier(
    n_estimators=100,      
    max_depth=None,         
    random_state=42,       
    n_jobs=-1               
)

rf_model.fit(X_train, y_train) #train model
y_pred_rf = rf_model.predict(X_test) #prediction
acc_rf = accuracy_score(y_test, y_pred_rf) #check the accuracy

print("Accuracy random forest")
print(f"Accuracy: {acc_rf:.4f}\n")

print("Report")
print(classification_report(y_test, y_pred_rf))

Accuracy random forest
Accuracy: 0.7579

Report
              precision    recall  f1-score   support

           0       0.79      0.70      0.74       863
           1       0.74      0.81      0.77       876

    accuracy                           0.76      1739
   macro avg       0.76      0.76      0.76      1739
weighted avg       0.76      0.76      0.76      1739



- Gradient Boosting

In [15]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

gb_model = GradientBoostingClassifier(
    n_estimators=100,      
    learning_rate=0.1,    
    max_depth=3,         
    random_state=42
)
gb_model.fit(X_train, y_train) #train the model
y_pred_gb = gb_model.predict(X_test) #prediction
acc_gb = accuracy_score(y_test, y_pred_gb) #accuracy

print("Accuracy")
print(f"Accuracy : {acc_gb:.4f}\n")

print("Report")
print(classification_report(y_test, y_pred_gb))

Accuracy
Accuracy : 0.7752

Report
              precision    recall  f1-score   support

           0       0.80      0.73      0.76       863
           1       0.76      0.82      0.79       876

    accuracy                           0.78      1739
   macro avg       0.78      0.77      0.77      1739
weighted avg       0.78      0.78      0.77      1739



- Adaptive Boosting

In [16]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report

adaboost_model = AdaBoostClassifier(
    n_estimators=100,     
    learning_rate=1.0,    
    random_state=42
)

adaboost_model.fit(X_train, y_train) #train model
y_pred_ada = adaboost_model.predict(X_test) #predictions
acc_ada = accuracy_score(y_test, y_pred_ada) #accuracy

print("Accuracy")
print(f"Accuracy: {acc_ada:.4f}\n")

print("Report")
print(classification_report(y_test, y_pred_ada))

Accuracy
Accuracy: 0.7556

Report
              precision    recall  f1-score   support

           0       0.76      0.75      0.75       863
           1       0.75      0.76      0.76       876

    accuracy                           0.76      1739
   macro avg       0.76      0.76      0.76      1739
weighted avg       0.76      0.76      0.76      1739



Which model is the best and why?

The best model is Gradient Boosting, because it has an accuracy of  0.7752